# SSE Association Sparsity and Separation Diagnostics

This notebook checks whether the association-analysis model frames are sparse or nearly separated. The global candidate/background balance can look healthy while individual windows, clades, predictor levels, or complete-case model frames still contain only one outcome class or very small minority-class counts.

The checks below use the same prepared frames as the association pipeline.

In [1]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sse_detection import lib as sselib

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

## Configuration

In [3]:
WINDOW_STRIDE = 2
VARIANT_ADJUSTER = "clade"
SPARSE_MIN_CLASS = 10
NEAR_DETERMINISTIC_RATE = 0.98
N_BINS = 10

MODEL_SETS = sselib.default_model_sets(
    variant_adjuster=VARIANT_ADJUSTER,
    window_adjustment="fixed_effects",
)

MIXING_FEATURES = list(sselib.DEFAULT_MIXING_FEATURES)
COMPOSITION_FEATURES = [spec["column"] for spec in sselib.COMPOSITION_SPECS]

MODEL_SETS, MIXING_FEATURES, COMPOSITION_FEATURES

({'primary': ['C(window_idx)', 'C(clade)'],
  'expanded': ['C(window_idx)',
   'C(clade)',
   'z_dz_cum_prop_sequenced',
   'z_dz_cum_incidence_per_capita',
   'z_dz_7d_test_positivity',
   'z_log1p_dz_cum_positive_tests']},
 ['sex_entropy_z',
  'age_entropy_z',
  'simd_entropy_z',
  'datazone_entropy_z',
  'local_authority_entropy_z',
  'urban_rural_entropy_z',
  'health_board_entropy_z',
  'vaccination_entropy_z'],
 ['sex',
  'age_band',
  'dz_simd_quintile',
  'dz_urban_rural_class',
  'dz_health_board'])

## Load Analysis Frames

In [4]:
frames = sselib.load_association_frames(
    output_dir=PROJECT_ROOT / "sse_detection" / "results" / "sse_outputs",
    variant_adjuster=VARIANT_ADJUSTER,
    group_by_clade=False,
    window_stride=WINDOW_STRIDE,
    run_composition=True,
)

node_df = frames.node_model_base.copy()
composition_df = frames.composition_base.copy()

display(
    pd.DataFrame(
        [
            {
                "frame": "node_model_base",
                "rows": len(node_df),
                "nodes": node_df["cluster_id"].nunique(),
                "background": int((node_df["candidate"] == 0).sum()),
                "candidate": int((node_df["candidate"] == 1).sum()),
            },
            {
                "frame": "composition_base",
                "rows": len(composition_df),
                "nodes": composition_df["cluster_id"].nunique(),
                "background": int((composition_df["candidate"] == 0).sum()),
                "candidate": int((composition_df["candidate"] == 1).sum()),
            },
        ]
    )
)

print(f"Minimum candidate cluster size threshold: {frames.min_candidate_size}")
display(frames.cluster_diagnostics)

,frame,rows,nodes,background,candidate
0,node_model_base,13059,13059,12445,614
1,composition_base,264139,13059,197299,66840


Minimum candidate cluster size threshold: 6


,cluster_col,n_rows,n_clusters,min_rows_per_cluster,median_rows_per_cluster,outcome_positive_clusters,outcome_varying_clusters,analysis_frame
0,cluster_id,264139,13059,1,9.0,614,0,composition
1,cluster_id,13059,13059,1,1.0,614,0,node_mixing


## Helper Functions

In [5]:
def normalise_group_cols(group_cols):
    return [group_cols] if isinstance(group_cols, str) else list(group_cols)


def outcome_balance(data, outcome="candidate"):
    counts = data[outcome].dropna().astype(int).value_counts().reindex([0, 1], fill_value=0)
    total = int(counts.sum())
    return pd.DataFrame(
        [
            {
                "background": int(counts.loc[0]),
                "candidate": int(counts.loc[1]),
                "total": total,
                "candidate_rate": counts.loc[1] / total if total else np.nan,
                "min_class": int(counts.min()) if total else 0,
            }
        ]
    )


def balance_by(
    data,
    group_cols,
    *,
    outcome="candidate",
    min_class_threshold=SPARSE_MIN_CLASS,
    near_rate=NEAR_DETERMINISTIC_RATE,
):
    group_cols = normalise_group_cols(group_cols)
    required = [*group_cols, outcome]
    d = data.dropna(subset=required).copy()
    if d.empty:
        return pd.DataFrame()

    d[outcome] = d[outcome].astype(int)
    counts = d.groupby([*group_cols, outcome], dropna=False).size().unstack(outcome, fill_value=0)
    for cls in [0, 1]:
        if cls not in counts.columns:
            counts[cls] = 0
    counts = counts[[0, 1]].rename(columns={0: "background", 1: "candidate"})
    out = counts.reset_index()
    out["total"] = out["background"] + out["candidate"]
    out["candidate_rate"] = np.where(out["total"].gt(0), out["candidate"] / out["total"], np.nan)
    out["min_class"] = out[["background", "candidate"]].min(axis=1)
    out["separated"] = out["min_class"].eq(0)
    out["sparse"] = out["min_class"].lt(min_class_threshold)
    out["near_deterministic"] = out["candidate_rate"].ge(near_rate) | out["candidate_rate"].le(1 - near_rate)
    return out.sort_values(
        ["separated", "sparse", "min_class", "total"],
        ascending=[False, False, True, False],
    ).reset_index(drop=True)


def summarise_balance(table, name):
    if table.empty:
        return {
            "check": name,
            "groups": 0,
            "separated_groups": 0,
            "sparse_groups": 0,
            "near_deterministic_groups": 0,
            "min_class_min": np.nan,
        }
    return {
        "check": name,
        "groups": len(table),
        "separated_groups": int(table["separated"].sum()),
        "sparse_groups": int(table["sparse"].sum()),
        "near_deterministic_groups": int(table["near_deterministic"].sum()),
        "min_class_min": int(table["min_class"].min()),
    }


def binned_balance(data, feature, *, q=N_BINS, outcome="candidate"):
    d = data[[feature, outcome]].dropna().copy()
    if d[feature].nunique(dropna=True) < 2:
        return pd.DataFrame()
    d[f"{feature}_bin"] = pd.qcut(d[feature], q=q, duplicates="drop")
    out = balance_by(d, f"{feature}_bin", outcome=outcome)
    out.insert(0, "feature", feature)
    return out


def complete_case_frame(data, predictors, adjusters):
    required = ["candidate", *predictors, *sselib.model_variables_from_terms(adjusters)]
    required = list(dict.fromkeys(required))
    missing = [col for col in required if col not in data.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    return data.dropna(subset=required).copy()


def model_case_summary(data, *, domain, model_set, predictor_set, predictors, adjusters, group_checks):
    d = complete_case_frame(data, predictors, adjusters)
    overall = outcome_balance(d).iloc[0]
    row = {
        "domain": domain,
        "model_set": model_set,
        "predictor_set": predictor_set,
        "predictors": "+".join(predictors),
        "n_rows": len(d),
        "background": int(overall["background"]),
        "candidate": int(overall["candidate"]),
        "candidate_rate": overall["candidate_rate"],
        "min_class": int(overall["min_class"]),
    }
    for check_name, group_cols in group_checks.items():
        group_cols = normalise_group_cols(group_cols)
        if not all(col in d.columns for col in group_cols):
            row[f"{check_name}_groups"] = np.nan
            row[f"{check_name}_separated"] = np.nan
            row[f"{check_name}_sparse"] = np.nan
            row[f"{check_name}_min_class"] = np.nan
            continue
        table = balance_by(d, group_cols)
        row[f"{check_name}_groups"] = len(table)
        row[f"{check_name}_separated"] = int(table["separated"].sum()) if not table.empty else 0
        row[f"{check_name}_sparse"] = int(table["sparse"].sum()) if not table.empty else 0
        row[f"{check_name}_min_class"] = int(table["min_class"].min()) if not table.empty else np.nan
    return row

## Overall Outcome Balance

In [6]:
display(outcome_balance(node_df).assign(frame="node_model_base"))
display(outcome_balance(composition_df).assign(frame="composition_base"))

,background,candidate,total,candidate_rate,min_class,frame
0,12445,614,13059,0.047017,614,node_model_base


,background,candidate,total,candidate_rate,min_class,frame
0,197299,66840,264139,0.253049,66840,composition_base


## Node-Level Strata Checks

These are the most relevant diagnostics for node-level mixing models. `window_x_clade` is not an explicit interaction in the default formula, but it is useful for spotting local data gaps.

In [7]:
node_strata_checks = {
    "window": "window_idx",
    "clade": "clade",
    "window_x_clade": ["window_idx", "clade"],
}

node_strata_tables = {}
node_strata_summary = []

for name, group_cols in node_strata_checks.items():
    table = balance_by(node_df, group_cols)
    node_strata_tables[name] = table
    node_strata_summary.append(summarise_balance(table, name))
    print(f"\n{name}")
    display(table.head(25))

node_strata_summary = pd.DataFrame(node_strata_summary)
display(node_strata_summary)


window


candidate,window_idx,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,61,42,0,42,0.000000,0,True,True,True
1,66,40,0,40,0.000000,0,True,True,True
2,63,36,0,36,0.000000,0,True,True,True
3,67,27,0,27,0.000000,0,True,True,True
4,3,13,0,13,0.000000,0,True,True,True
5,2,3,0,3,0.000000,0,True,True,True
6,1,1,0,1,0.000000,0,True,True,True
7,6,54,1,55,0.018182,1,False,True,True
8,64,51,1,52,0.019231,1,False,True,True
9,58,47,1,48,0.020833,1,False,True,False



clade


candidate,clade,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,20B,61,0,61,0.000000,0,True,True,True
1,21I,49,0,49,0.000000,0,True,True,True
2,20A,37,0,37,0.000000,0,True,True,True
3,22D,33,0,33,0.000000,0,True,True,True
4,23C,19,0,19,0.000000,0,True,True,True
5,21A,15,0,15,0.000000,0,True,True,True
6,22F,3,0,3,0.000000,0,True,True,True
7,19B,2,0,2,0.000000,0,True,True,True
8,20D,1,0,1,0.000000,0,True,True,True
9,21D,1,0,1,0.000000,0,True,True,True



window_x_clade


candidate,window_idx,clade,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,25,20I,33,0,33,0.0,0,True,True,True
1,66,22E,22,0,22,0.0,0,True,True,True
2,63,22E,21,0,21,0.0,0,True,True,True
3,61,22E,20,0,20,0.0,0,True,True,True
4,53,22A,18,0,18,0.0,0,True,True,True
5,14,20E,17,0,17,0.0,0,True,True,True
6,51,22C,17,0,17,0.0,0,True,True,True
7,60,22E,17,0,17,0.0,0,True,True,True
8,61,22B,17,0,17,0.0,0,True,True,True
9,13,20I,14,0,14,0.0,0,True,True,True


,check,groups,separated_groups,sparse_groups,near_deterministic_groups,min_class_min
0,window,67,7,41,9,0
1,clade,21,10,15,12,0
2,window_x_clade,206,128,180,128,0


## Clade-Group Sensitivity Balance

In [8]:
node_with_clade_group = sselib.add_clade_group(node_df, source_col="clade", target_col="clade_group")
clade_group_balance = balance_by(node_with_clade_group, "clade_group")
display(clade_group_balance)

clade_window_balance = balance_by(node_with_clade_group, ["clade_group", "window_idx"])
display(clade_window_balance.head(40))

candidate,clade_group,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,20B,61,0,61,0.000000,0,True,True,True
1,21I (Delta),49,0,49,0.000000,0,True,True,True
2,20A,37,0,37,0.000000,0,True,True,True
3,22A (Omicron),123,2,125,0.016000,2,False,True,True
4,22E (Omicron),193,3,196,0.015306,3,False,True,True
5,Other,129,3,132,0.022727,3,False,True,False
6,22C (Omicron),54,3,57,0.052632,3,False,True,False
7,20E (EU1),334,12,346,0.034682,12,False,False,False
8,22B (Omicron),913,38,951,0.039958,38,False,False,False
9,20I (Alpha),1442,67,1509,0.044400,67,False,False,False


candidate,clade_group,window_idx,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,20I (Alpha),25,33,0,33,0.0,0,True,True,True
1,22E (Omicron),66,22,0,22,0.0,0,True,True,True
2,22E (Omicron),63,21,0,21,0.0,0,True,True,True
3,22E (Omicron),61,20,0,20,0.0,0,True,True,True
4,22A (Omicron),53,18,0,18,0.0,0,True,True,True
5,20E (EU1),14,17,0,17,0.0,0,True,True,True
6,22B (Omicron),61,17,0,17,0.0,0,True,True,True
7,22C (Omicron),51,17,0,17,0.0,0,True,True,True
8,22E (Omicron),60,17,0,17,0.0,0,True,True,True
9,Other,67,15,0,15,0.0,0,True,True,True


## Composition Predictor Level Checks

These checks use the sequence-level composition frame. The outcome is still the cluster/node-level candidate label.

In [9]:
composition_level_tables = []
composition_level_summary = []

for spec in sselib.COMPOSITION_SPECS:
    col = spec["column"]
    table = balance_by(composition_df, col)
    table.insert(0, "predictor", spec["name"])
    composition_level_tables.append(table)
    composition_level_summary.append(summarise_balance(table, spec["name"]))
    print(f"\n{spec['label']} ({col})")
    display(table.head(25))

composition_level_balance = pd.concat(composition_level_tables, ignore_index=True)
composition_level_summary = pd.DataFrame(composition_level_summary)
display(composition_level_summary)


Sex (sex)

candidate,predictor,sex,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,sex,Male,92982,31509,124491,0.253103,31509,False,False,False
1,sex,Female,104317,35331,139648,0.253000,35331,False,False,False



Age band (age_band)


candidate,predictor,age_band,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,age_band,70-74,4377,1393,5770,0.241421,1393,False,False,False
1,age_band,00-04,4414,1398,5812,0.240537,1398,False,False,False
2,age_band,65-69,5208,1705,6913,0.246637,1705,False,False,False
3,age_band,60-64,9221,3083,12304,0.250569,3083,False,False,False
4,age_band,75+,11473,3190,14663,0.217554,3190,False,False,False
5,age_band,05-09,11560,3380,14940,0.226238,3380,False,False,False
6,age_band,55-59,12415,4091,16506,0.247849,4091,False,False,False
7,age_band,10-14,14063,4335,18398,0.235623,4335,False,False,False
8,age_band,45-49,13123,4438,17561,0.252719,4438,False,False,False
9,age_band,50-54,13516,4661,18177,0.256423,4661,False,False,False



SIMD quintile (dz_simd_quintile)


...


Urban/rural class (dz_urban_rural_class)


candidate,predictor,dz_urban_rural_class,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,urban_rural_class,Remote Small Towns,5475,1862,7337,0.253782,1862,False,False,False
1,urban_rural_class,Remote Rural,6811,2292,9103,0.251785,2292,False,False,False
2,urban_rural_class,Accessible Small Towns,16216,5239,21455,0.244186,5239,False,False,False
3,urban_rural_class,Accessible Rural,19123,6329,25452,0.248664,6329,False,False,False
4,urban_rural_class,Other Urban Areas,74902,25545,100447,0.254313,25545,False,False,False
5,urban_rural_class,Large Urban Areas,74772,25573,100345,0.254851,25573,False,False,False



Health board (dz_health_board)


candidate,predictor,dz_health_board,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,health_board,Western Isles,184,133,317,0.419558,133,False,False,False
1,health_board,Shetland,631,200,831,0.240674,200,False,False,False
2,health_board,Orkney,609,246,855,0.287719,246,False,False,False
3,health_board,Borders,3233,1011,4244,0.238219,1011,False,False,False
4,health_board,Dumfries and Galloway,4238,1393,5631,0.247381,1393,False,False,False
5,health_board,Highland,8745,3223,11968,0.269301,3223,False,False,False
6,health_board,Fife,13395,4160,17555,0.236970,4160,False,False,False
7,health_board,Forth Valley,11838,4263,16101,0.264766,4263,False,False,False
8,health_board,Ayrshire and Arran,13372,4748,18120,0.262031,4748,False,False,False
9,health_board,Grampian,17345,5295,22640,0.233878,5295,False,False,False


,check,groups,separated_groups,sparse_groups,near_deterministic_groups,min_class_min
0,sex,2,0,0,0,31509
1,age_band,16,0,0,0,1393
2,simd_quintile,5,0,0,0,11922
3,urban_rural_class,6,0,0,0,1862
4,health_board,14,0,0,0,133


## Mixing Predictor Decile Checks

Continuous predictors cannot be checked by exact levels. Binning them into deciles gives a practical screen for near-deterministic outcome regions.

In [10]:
mixing_bin_tables = []
mixing_bin_summary = []

for feature in MIXING_FEATURES:
    table = binned_balance(node_df, feature)
    mixing_bin_tables.append(table)
    mixing_bin_summary.append(summarise_balance(table, feature))
    print(f"\n{feature}")
    display(table)

mixing_bin_balance = pd.concat(mixing_bin_tables, ignore_index=True)
mixing_bin_summary = pd.DataFrame(mixing_bin_summary)
display(mixing_bin_summary)


sex_entropy_z


candidate,feature,sex_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,sex_entropy_z,"(0.602, 0.648]",1273,28,1301,0.021522,28,False,False,False
1,sex_entropy_z,"(0.503, 0.602]",1251,46,1297,0.035466,46,False,False,False
2,sex_entropy_z,"(0.191, 0.341]",1290,49,1339,0.036594,49,False,False,False
3,sex_entropy_z,"(0.648, 0.693]",1240,51,1291,0.039504,51,False,False,False
4,sex_entropy_z,"(-1.402, -0.574]",1245,58,1303,0.044513,58,False,False,False
5,sex_entropy_z,"(0.341, 0.503]",1194,62,1256,0.049363,62,False,False,False
6,sex_entropy_z,"(-0.574, -0.135]",1239,64,1303,0.049117,64,False,False,False
7,sex_entropy_z,"(-0.135, 0.191]",1212,72,1284,0.056075,72,False,False,False
8,sex_entropy_z,"(-53.836999999999996, -1.402]",1212,85,1297,0.065536,85,False,False,False
9,sex_entropy_z,"(0.693, 1.828]",1197,99,1296,0.076389,99,False,False,False



age_entropy_z


candidate,feature,age_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,age_entropy_z,"(-0.172, 0.114]",1265,37,1302,0.028418,37,False,False,False
1,age_entropy_z,"(-1.28, -0.886]",1249,45,1294,0.034776,45,False,False,False
2,age_entropy_z,"(-0.492, -0.172]",1244,52,1296,0.040123,52,False,False,False
3,age_entropy_z,"(0.479, 0.943]",1243,54,1297,0.041635,54,False,False,False
4,age_entropy_z,"(0.114, 0.479]",1231,60,1291,0.046476,60,False,False,False
5,age_entropy_z,"(-1.845, -1.28]",1236,63,1299,0.048499,63,False,False,False
6,age_entropy_z,"(-0.886, -0.492]",1232,65,1297,0.050116,65,False,False,False
7,age_entropy_z,"(0.943, 3.116]",1227,70,1297,0.053971,70,False,False,False
8,age_entropy_z,"(-2.803, -1.845]",1224,73,1297,0.056284,73,False,False,False
9,age_entropy_z,"(-25.427, -2.803]",1202,95,1297,0.073246,95,False,False,False



simd_entropy_z


...


health_board_entropy_z


candidate,feature,health_board_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,health_board_entropy_z,"(-3.001, -2.329]",1258,41,1299,0.031563,41,False,False,False
1,health_board_entropy_z,"(-5.265, -4.466]",1255,42,1297,0.032382,42,False,False,False
2,health_board_entropy_z,"(-3.667, -3.001]",1253,43,1296,0.033179,43,False,False,False
3,health_board_entropy_z,"(-6.184, -5.265]",1240,46,1286,0.035770,46,False,False,False
4,health_board_entropy_z,"(-4.466, -3.667]",1248,49,1297,0.037779,49,False,False,False
5,health_board_entropy_z,"(-2.329, -1.547]",1246,51,1297,0.039322,51,False,False,False
6,health_board_entropy_z,"(-1.547, -0.632]",1240,54,1294,0.041731,54,False,False,False
7,health_board_entropy_z,"(-7.611, -6.184]",1242,65,1307,0.049732,65,False,False,False
8,health_board_entropy_z,"(-0.632, 3.935]",1232,65,1297,0.050116,65,False,False,False
9,health_board_entropy_z,"(-30.562, -7.611]",1139,158,1297,0.121820,158,False,False,False



vaccination_entropy_z


candidate,feature,vaccination_entropy_z_bin,background,candidate,total,candidate_rate,min_class,separated,sparse,near_deterministic
0,vaccination_entropy_z,"(-0.967, -0.649]",1220,45,1265,0.035573,45,False,False,False
1,vaccination_entropy_z,"(0.605, 0.81]",1219,46,1265,0.036364,46,False,False,False
2,vaccination_entropy_z,"(0.81, 1.149]",1215,49,1264,0.038766,49,False,False,False
3,vaccination_entropy_z,"(-0.649, -0.32]",1211,54,1265,0.042688,54,False,False,False
4,vaccination_entropy_z,"(-1.417, -0.967]",1207,57,1264,0.045095,57,False,False,False
5,vaccination_entropy_z,"(0.0646, 0.38]",1202,62,1264,0.049051,62,False,False,False
6,vaccination_entropy_z,"(0.38, 0.605]",1200,64,1264,0.050633,64,False,False,False
7,vaccination_entropy_z,"(-0.32, 0.0646]",1197,67,1264,0.053006,67,False,False,False
8,vaccination_entropy_z,"(-12.841999999999999, -1.417]",1193,72,1265,0.056917,72,False,False,False
9,vaccination_entropy_z,"(1.149, 4.552]",1176,89,1265,0.070356,89,False,False,False


,check,groups,separated_groups,sparse_groups,near_deterministic_groups,min_class_min
0,sex_entropy_z,10,0,0,0,28
1,age_entropy_z,10,0,0,0,37
2,simd_entropy_z,10,0,0,0,46
3,datazone_entropy_z,10,0,1,1,8
4,local_authority_entropy_z,10,0,0,0,39
5,urban_rural_entropy_z,10,0,0,0,38
6,health_board_entropy_z,10,0,0,0,41
7,vaccination_entropy_z,10,0,0,0,45


## Complete-Case Model Frame Checks

These rows summarize the complete-case data available to each default main-analysis model. They are a compact way to see whether adding adjusters creates separated or sparse fixed-effect strata.

In [11]:
model_rows = []
node_group_checks = {
    "window": "window_idx",
    "clade": "clade",
    "window_x_clade": ["window_idx", "clade"],
}
composition_group_checks = {
    "window": "window_idx",
    "clade": "clade",
    "window_x_clade": ["window_idx", "clade"],
}

for model_set, adjusters in MODEL_SETS.items():
    for feature in MIXING_FEATURES:
        model_rows.append(
            model_case_summary(
                node_df,
                domain="node_mixing",
                model_set=model_set,
                predictor_set="single",
                predictors=[feature],
                adjusters=adjusters,
                group_checks=node_group_checks,
            )
        )
    model_rows.append(
        model_case_summary(
            node_df,
            domain="node_mixing",
            model_set=model_set,
            predictor_set="joint",
            predictors=MIXING_FEATURES,
            adjusters=adjusters,
            group_checks=node_group_checks,
        )
    )

    for spec in sselib.COMPOSITION_SPECS:
        model_rows.append(
            model_case_summary(
                composition_df,
                domain="composition",
                model_set=model_set,
                predictor_set="single",
                predictors=[spec["column"]],
                adjusters=adjusters,
                group_checks=composition_group_checks,
            )
        )
    model_rows.append(
        model_case_summary(
            composition_df,
            domain="composition",
            model_set=model_set,
            predictor_set="joint",
            predictors=COMPOSITION_FEATURES,
            adjusters=adjusters,
            group_checks=composition_group_checks,
        )
    )

model_complete_case_summary = pd.DataFrame(model_rows)
display(
    model_complete_case_summary.sort_values(
        ["domain", "model_set", "predictor_set", "predictors"]
    )
)

,domain,model_set,predictor_set,predictors,n_rows,background,candidate,candidate_rate,min_class,window_groups,window_separated,window_sparse,window_min_class,clade_groups,clade_separated,clade_sparse,clade_min_class,window_x_clade_groups,window_x_clade_separated,window_x_clade_sparse,window_x_clade_min_class
29,composition,expanded,joint,sex+age_band+dz_simd_quintile+dz_urban_rural_c...,264119,197279,66840,0.253068,66840,67,7,9,0,21,10,10,0,206,128,130,0
25,composition,expanded,single,age_band,264119,197279,66840,0.253068,66840,67,7,9,0,21,10,10,0,206,128,130,0
28,composition,expanded,single,dz_health_board,264119,197279,66840,0.253068,66840,67,7,9,0,21,10,10,0,206,128,130,0
26,composition,expanded,single,dz_simd_quintile,264119,197279,66840,0.253068,66840,67,7,9,0,21,10,10,0,206,128,130,0
27,composition,expanded,single,dz_urban_rural_class,264119,197279,66840,0.253068,66840,67,7,9,0,21,10,10,0,206,128,130,0
24,composition,expanded,single,sex,264119,197279,66840,0.253068,66840,67,7,9,0,21,10,10,0,206,128,130,0
14,composition,primary,joint,sex+age_band+dz_simd_quintile+dz_urban_rural_c...,264139,197299,66840,0.253049,66840,67,7,9,0,21,10,10,0,206,128,130,0
10,composition,primary,single,age_band,264139,197299,66840,0.253049,66840,67,7,9,0,21,10,10,0,206,128,130,0
13,composition,primary,single,dz_health_board,264139,197299,66840,0.253049,66840,67,7,9,0,21,10,10,0,206,128,130,0
11,composition,primary,single,dz_simd_quintile,264139,197299,66840,0.253049,66840,67,7,9,0,21,10,10,0,206,128,130,0


## Optional Standard-GLM Stress Test

Firth models are used because they are more stable under sparse or separated data. This optional ordinary-GLM fit is a diagnostic: very large coefficients, very large standard errors, warnings, or fitted probabilities near 0/1 indicate separation pressure.

In [12]:
OPTIONAL_FORMULA = "candidate ~ sex_entropy_z + C(window_idx) + C(clade)"
OPTIONAL_REQUIRED = ["candidate", "sex_entropy_z", "window_idx", "clade"]

stress_df = node_df.dropna(subset=OPTIONAL_REQUIRED).copy()
print(f"Rows in stress-test model frame: {len(stress_df):,}")

try:
    stress_result = smf.glm(
        OPTIONAL_FORMULA,
        data=stress_df,
        family=sm.families.Binomial(),
        missing="raise",
    ).fit(maxiter=100)

    stress_terms = pd.DataFrame(
        {
            "estimate": stress_result.params,
            "std_error": stress_result.bse,
            "z": stress_result.tvalues,
            "p_value": stress_result.pvalues,
        }
    )
    pred = stress_result.predict(stress_df)
    print(
        "Near-zero/one fitted probability share:",
        float(((pred < 1e-6) | (pred > 1 - 1e-6)).mean()),
    )
    display(stress_terms.assign(abs_estimate=stress_terms["estimate"].abs()).sort_values("abs_estimate", ascending=False).head(20))
    display(stress_terms.sort_values("std_error", ascending=False).head(20))
except Exception as exc:
    print(f"Ordinary GLM stress test failed: {type(exc).__name__}: {exc}")

Rows in stress-test model frame: 12,967


Near-zero/one fitted probability share: 0.025140741883242074


,estimate,std_error,z,p_value,abs_estimate
C(clade)[T.23A],27.503483,251725.853685,0.000109,0.999913,27.503483
C(window_idx)[T.67],-26.840465,365078.454803,-0.000074,0.999941,26.840465
C(window_idx)[T.66],-26.476549,363751.674044,-0.000073,0.999942,26.476549
Intercept,-26.008968,438632.983006,-0.000059,0.999953,26.008968
C(window_idx)[T.61],-25.860209,363750.639846,-0.000071,0.999943,25.860209
C(window_idx)[T.63],-25.748178,364026.620073,-0.000071,0.999944,25.748178
C(clade)[T.22E],25.491312,251725.853681,0.000101,0.999919,25.491312
C(clade)[T.22B],25.481789,251725.853679,0.000101,0.999919,25.481789
C(clade)[T.22C],25.455902,251725.853679,0.000101,0.999919,25.455902
C(clade)[T.21L],25.090747,251725.853679,0.000100,0.999920,25.090747


,estimate,std_error,z,p_value
Intercept,-26.008968,438632.983006,-5.929551e-05,0.999953
C(clade)[T.20D],-0.846821,436108.023789,-1.941770e-06,0.999998
C(clade)[T.21D],0.069188,436108.023644,1.586478e-07,1.000000
C(window_idx)[T.2],-0.050380,411330.616797,-1.224814e-07,1.000000
C(window_idx)[T.3],-21.624469,368752.557977,-5.864222e-05,0.999953
C(window_idx)[T.67],-26.840465,365078.454803,-7.351972e-05,0.999941
C(window_idx)[T.63],-25.748178,364026.620073,-7.073158e-05,0.999944
C(window_idx)[T.66],-26.476549,363751.674044,-7.278743e-05,0.999942
C(window_idx)[T.61],-25.860209,363750.639846,-7.109323e-05,0.999943
C(window_idx)[T.64],-3.176160,359211.626672,-8.842031e-06,0.999993


## Export Diagnostics

In [13]:
DIAGNOSTIC_DIR = PROJECT_ROOT / "sse_detection" / "results" / "separation_diagnostics"
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)

node_strata_summary_df = (
    node_strata_summary
    if isinstance(node_strata_summary, pd.DataFrame)
    else pd.DataFrame(node_strata_summary)
)
node_strata_summary_df.to_csv(DIAGNOSTIC_DIR / "node_strata_summary.csv", index=False)
pd.concat(
    [table.assign(check=name) for name, table in node_strata_tables.items()],
    ignore_index=True,
).to_csv(DIAGNOSTIC_DIR / "node_strata_balance.csv", index=False)
clade_group_balance.to_csv(DIAGNOSTIC_DIR / "clade_group_balance.csv", index=False)
clade_window_balance.to_csv(DIAGNOSTIC_DIR / "clade_window_balance.csv", index=False)
composition_level_summary.to_csv(DIAGNOSTIC_DIR / "composition_level_summary.csv", index=False)
composition_level_balance.to_csv(DIAGNOSTIC_DIR / "composition_level_balance.csv", index=False)
mixing_bin_summary.to_csv(DIAGNOSTIC_DIR / "mixing_bin_summary.csv", index=False)
mixing_bin_balance.to_csv(DIAGNOSTIC_DIR / "mixing_bin_balance.csv", index=False)
model_complete_case_summary.to_csv(DIAGNOSTIC_DIR / "model_complete_case_summary.csv", index=False)

print(f"Saved diagnostics to: {DIAGNOSTIC_DIR.relative_to(PROJECT_ROOT)}")

Saved diagnostics to: sse_detection/results/separation_diagnostics
